# LaRS Spatial OOD Analysis

**Extension to the whole-image embedding analysis in `analysis.ipynb`.**

Extracts per-patch spatial token embeddings from NVIDIA RADIO-H, builds a FAISS
flat index over a uniform subsample of training patches (1-in-10, ~1.4 M vectors,
fits in RAM), then computes per-patch OOD scores for test and val images. The result
is a spatial heatmap per image showing which regions are underrepresented in training.

| Section | Content |
|---------|--------|
| 1 | Setup — imports, paths, GPU/FAISS check |
| 2 | Build FAISS index from subsampled training patches |
| 3 | Compute OOD heatmaps for test + val |
| 4 | Distribution of spatial OOD scores |
| 5 | Top-20 most OOD test images with heatmap overlays |
| 6 | Aggregate spatial OOD patterns |
| 7 | Report generation |

**Memory budget note**: all 14 M training patches (68 GB) are stored in
`train_patches.dat` (memmap). The FAISS index uses a uniform 1-in-10 subsample
(~1.4 M × 1280 × 4 B ≈ 7 GB), which fits comfortably in the 62 GB RAM available.

## Section 1 — Setup

In [1]:
import os, sys, time, sqlite3, random, logging
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import faiss

# ---- Paths ----
ANALYSIS_DIR   = Path('/home/farms/macvi2026-farms-team/challenges/common/lars-analysis')
SPATIAL_DIR    = ANALYSIS_DIR / 'spatial_ood'
HEATMAPS_DIR   = SPATIAL_DIR / 'heatmaps'
FIGURES_DIR    = SPATIAL_DIR / 'figures'
DATASET_ROOT   = Path('/home/farms/data/lars_dataset/img_dir')
DB_PATH        = ANALYSIS_DIR / 'embeddings.db'
LOG_PATH       = ANALYSIS_DIR / 'logs' / 'spatial_ood.log'

# Intermediate artifacts (not committed)
PATCHES_DAT      = SPATIAL_DIR / 'train_patches.dat'       # all 14 M patches (68 GB)
PATCH_META_NPY   = SPATIAL_DIR / 'train_patch_meta.npy'    # metadata for all patches
TOKEN_COUNT_NPY  = SPATIAL_DIR / 'train_token_counts.npy'  # tokens per training image
FAISS_INDEX      = SPATIAL_DIR / 'faiss_index.bin'         # subsampled flat index (~7 GB)

# Report
REPORT_PATH    = SPATIAL_DIR.parent / 'reports' / 'spatial_ood_report.pdf'
REPORT_GEN     = SPATIAL_DIR / 'generate_spatial_report.py'

MODEL_NAME   = 'nvidia/C-RADIOv3-H'
EMBED_DIM    = 1280
COMMIT_EVERY = 50

# Fraction of training patches to keep in the FAISS index.
# 1/10 of 14 M = ~1.4 M vectors @ 1280×float32 = 7.2 GB — fits in 62 GB RAM.
SUBSAMPLE_RATE = 10   # keep every Nth patch

# Chunk size (query rows) for the OOM-fallback path in Phase 2.
# 512 rows x 1,408,492 cols x 2 B = ~1.44 GB intermediate -- safe on 48 GB VRAM.
# Result is bit-identical to the full-matmul fast path (per-row max is independent).
QUERY_CHUNK_OOM = 512

# ---- Logging ----
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.getLogger('spatial_ood').handlers.clear()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH, mode='a'),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger('spatial_ood')
HEATMAPS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---- GPU / VRAM ----
log.info('PyTorch %s  CUDA=%s', torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    log.info('  GPU 0: %s  %.1f GB', props.name, props.total_memory/1024**3)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log.info('Device: %s', DEVICE)

# ---- FAISS ----
FAISS_GPU = faiss.get_num_gpus() > 0
log.info('FAISS GPU: %s  (num_gpus=%d)', FAISS_GPU, faiss.get_num_gpus())

# ---- Helpers ----
def get_image_list(split):
    names = (DATASET_ROOT / split / 'image_list.txt').read_text().strip().splitlines()
    return [{'name': n, 'split': split,
              'sequence': n.rsplit('_',1)[0] if '_' in n else n,
              'path': DATASET_ROOT / split / 'images' / (n+'.jpg')}
             for n in names]

def run_radio(model, img_pil, device):
    W, H = img_pil.size
    th, tw = model.get_nearest_supported_resolution(H, W)
    img_t = (torch.from_numpy(np.array(img_pil.resize((tw,th),Image.BICUBIC))
                               .astype(np.float32)/255.0)
              .permute(2,0,1).unsqueeze(0).to(device))
    with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16):
        out = model(img_t)
    if isinstance(out,(list,tuple)) and len(out)>=2:
        _, sp = out[0], out[1]
    elif hasattr(out,'spatial_features'):
        sp = out.spatial_features
    else:
        sp = out
    return sp.squeeze(0).float().cpu().numpy(), tw, th

2026-03-02 23:39:18,196 INFO PyTorch 2.5.1+cu121  CUDA=True


2026-03-02 23:39:18,219 INFO   GPU 0: NVIDIA RTX A6000  47.4 GB


2026-03-02 23:39:18,220 INFO Device: cuda


2026-03-02 23:39:18,221 INFO FAISS GPU: False  (num_gpus=0)


## Section 2 — Build FAISS Index from Subsampled Training Patches

**Skipped if `faiss_index.bin` already exists.**

The full memmap `train_patches.dat` (68 GB, ~14 M patches) was built in a previous
run and is already on disk. This section reads every 10th patch from it (~1.4 M
vectors, ~7 GB) and builds a `IndexFlatIP` (exact cosine similarity, no training
required). Flat index guarantees exact nearest-neighbour results for OOD scoring.

In [2]:
# Read token counts to know the memmap shape
token_counts      = np.load(TOKEN_COUNT_NPY)
valid_counts      = np.maximum(token_counts, 0)
total_patches_all = int(valid_counts.sum())
log.info('Total patches in memmap: %d (%.1f GB)',
         total_patches_all, total_patches_all * EMBED_DIM * 4 / 1e9)

if FAISS_INDEX.exists():
    log.info('faiss_index.bin exists — loading to verify.')
    idx_check = faiss.read_index(str(FAISS_INDEX))
    total_train_patches = total_patches_all   # still needed for metadata
    log.info('Index has %d vectors — skipping rebuild.', idx_check.ntotal)
    del idx_check
else:
    log.info('=== Building FAISS flat index from 1-in-%d subsample ===', SUBSAMPLE_RATE)
    t0 = time.time()

    # Open memmap read-only
    mm = np.memmap(PATCHES_DAT, dtype='float32', mode='r',
                   shape=(total_patches_all, EMBED_DIM))

    # Pick every SUBSAMPLE_RATE-th row
    sub_idx   = np.arange(0, total_patches_all, SUBSAMPLE_RATE)
    n_sub     = len(sub_idx)
    ram_gb    = n_sub * EMBED_DIM * 4 / 1e9
    log.info('Subsampled vectors: %d  (%.2f GB in RAM)', n_sub, ram_gb)

    # Read subsample in chunks to avoid a single huge memcpy
    CHUNK     = 200_000
    sub_data  = np.empty((n_sub, EMBED_DIM), dtype=np.float32)
    for ci, start in enumerate(tqdm(range(0, n_sub, CHUNK), desc='Reading subsample')):
        end = min(start + CHUNK, n_sub)
        sub_data[start:end] = mm[sub_idx[start:end]]
    del mm
    log.info('Subsample read in %.1fs', time.time() - t0)

    # Build flat IP index — no training required, exact cosine sim on L2-normed vectors
    log.info('Building IndexFlatIP (%d vectors × %d dims)...', n_sub, EMBED_DIM)
    t1    = time.time()
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(sub_data)
    log.info('Index built in %.1fs, ntotal=%d', time.time() - t1, index.ntotal)
    del sub_data

    faiss.write_index(index, str(FAISS_INDEX))
    del index
    log.info('FAISS index saved: %s  (%.2f GB)',
             FAISS_INDEX, FAISS_INDEX.stat().st_size / 1e9)

    total_train_patches = total_patches_all
    log.info('=== Phase 1 (FAISS build) complete: %.1f min ===', (time.time()-t0)/60)

log.info('Training patch index ready.  Subsampled vectors in index: 1-in-%d of %d',
         SUBSAMPLE_RATE, total_train_patches)

2026-03-02 23:39:18,238 INFO Total patches in memmap: 14084920 (72.1 GB)


2026-03-02 23:39:18,241 INFO faiss_index.bin exists — loading to verify.


2026-03-02 23:39:26,849 INFO Index has 1408492 vectors — skipping rebuild.


2026-03-02 23:39:27,536 INFO Training patch index ready.  Subsampled vectors in index: 1-in-10 of 14084920


## Section 3 — Compute Spatial OOD Heatmaps

**Skipped if `heatmaps/` already has the expected number of files.**

For each test and val image: run RADIO-H, L2-normalise tokens, compute max cosine
similarity per patch via PyTorch GPU matmul against the 1-in-10 subsampled training
vectors, reshape to grid, save as `.npy`.

OOM fallback: if a single all-patches matmul raises `torch.cuda.OutOfMemoryError`
(can happen for very high-resolution images with thousands of patches), the query
is automatically retried in `QUERY_CHUNK_OOM`-row chunks. The result is
bit-identical to the fast path since per-row max is independent of other rows.

In [3]:
test_records = get_image_list('test')
val_records  = get_image_list('val')
eval_records = test_records + val_records
n_eval       = len(eval_records)

existing = list(HEATMAPS_DIR.glob('*.npy'))
log.info('Existing heatmaps: %d / %d', len(existing), n_eval)

if len(existing) >= n_eval:
    log.info('Heatmaps complete — skipping Phase 2.')
else:
    log.info('=== Phase 2: Computing spatial OOD heatmaps (GPU matmul) ===')
    t2 = time.time()

    # ---- Load subsampled training vectors onto GPU as float16 ----
    # We re-read from the memmap (same 1-in-SUBSAMPLE_RATE selection used to build
    # the FAISS index) so that we don't need to parse the binary faiss_index.bin.
    token_counts_p2   = np.load(TOKEN_COUNT_NPY)
    total_patches_p2  = int(np.maximum(token_counts_p2, 0).sum())
    sub_idx_p2        = np.arange(0, total_patches_p2, SUBSAMPLE_RATE)
    n_sub_p2          = len(sub_idx_p2)
    log.info('Reading %d subsampled training vectors from memmap...', n_sub_p2)
    t_load = time.time()
    mm_p2       = np.memmap(PATCHES_DAT, dtype='float32', mode='r',
                             shape=(total_patches_p2, EMBED_DIM))
    # Read in chunks to avoid a single 7 GB contiguous allocation
    CHUNK_LOAD  = 200_000
    sub_f32     = np.empty((n_sub_p2, EMBED_DIM), dtype=np.float32)
    for ci, start in enumerate(tqdm(range(0, n_sub_p2, CHUNK_LOAD), desc='Load subsample')):
        end = min(start + CHUNK_LOAD, n_sub_p2)
        sub_f32[start:end] = mm_p2[sub_idx_p2[start:end]]
    del mm_p2
    log.info('Subsample loaded in %.1fs', time.time() - t_load)

    # Move to GPU as float16 — 1.4 M × 1280 × 2 B ≈ 3.6 GB on the GPU
    log.info('Moving training vectors to GPU (float16)...')
    train_gpu = torch.from_numpy(sub_f32).to(DEVICE).half()   # (N_sub, 1280)
    del sub_f32
    log.info('Training vectors on GPU: shape=%s  VRAM=%.1f GB',
             list(train_gpu.shape), train_gpu.numel()*2/1e9)

    # ---- Load RADIO-H ----
    log.info('Loading RADIO-H...')
    from transformers import AutoModel
    radio_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
    radio_model = radio_model.to(DEVICE).eval()
    log.info('Model loaded.')

    done    = {p.stem for p in existing}
    pending = [r for r in eval_records if r['name'] not in done]
    log.info('Pending: %d images', len(pending))

    with tqdm(total=len(pending), desc='Phase 2: Heatmaps') as pbar:
        for i, rec in enumerate(pending):
            try:
                img_pil = Image.open(rec['path']).convert('RGB')
                sp, pw, ph = run_radio(radio_model, img_pil, DEVICE)
                # L2-normalise query patches
                norms = np.linalg.norm(sp, axis=1, keepdims=True) + 1e-8
                sp_n  = (sp / norms).astype(np.float32)
                # GPU matmul: (n_patches, 1280) @ (1280, N_sub) → (n_patches, N_sub)
                # .max(dim=1) gives per-patch max cosine similarity.
                # Fast path: all patches in one shot.
                try:
                    sp_gpu  = torch.from_numpy(sp_n).to(DEVICE).half()
                    sims    = (sp_gpu @ train_gpu.T).max(dim=1).values
                    sims_np = sims.float().cpu().numpy()
                    del sp_gpu, sims
                except torch.cuda.OutOfMemoryError:
                    # Chunked fallback -- result is bit-identical to the fast path:
                    # per-row max is independent of other rows, so chunking the
                    # query matrix does not change the output.
                    torch.cuda.empty_cache()
                    log.warning('OOM %s (%d patches) -- retrying with %d-row chunks',
                                rec['name'], len(sp_n), QUERY_CHUNK_OOM)
                    sims_np = np.empty(len(sp_n), dtype=np.float32)
                    for ci in range(0, len(sp_n), QUERY_CHUNK_OOM):
                        chunk = torch.from_numpy(
                            sp_n[ci:ci+QUERY_CHUNK_OOM]).to(DEVICE).half()
                        chunk_sims = (chunk @ train_gpu.T).max(dim=1).values
                        sims_np[ci:ci+QUERY_CHUNK_OOM] = chunk_sims.float().cpu().numpy()
                        del chunk, chunk_sims
                    torch.cuda.empty_cache()
                    log.info('Chunked fallback complete for %s', rec['name'])
                heatmap = sims_np.reshape(ph // 16, pw // 16).astype(np.float32)
                np.save(HEATMAPS_DIR / f'{rec["name"]}.npy', heatmap)
            except Exception as e:
                log.warning('ERROR %s: %s', rec['name'], e)
            pbar.update(1)
            pbar.set_postfix({'img': f'{i+1}/{len(pending)}', 'split': rec['split']})
            if (i + 1) % COMMIT_EVERY == 0:
                log.info('  Heatmap progress: %d/%d  elapsed=%.1fmin',
                         i+1, len(pending), (time.time()-t2)/60)

    del radio_model, train_gpu
    torch.cuda.empty_cache()
    log.info('=== Phase 2 complete: %.1f min ===', (time.time()-t2)/60)

n_done = len(list(HEATMAPS_DIR.glob('*.npy')))
log.info('Total heatmap files: %d', n_done)

2026-03-02 23:39:27,583 INFO Existing heatmaps: 1401 / 1401


2026-03-02 23:39:27,585 INFO Heatmaps complete — skipping Phase 2.


2026-03-02 23:39:27,591 INFO Total heatmap files: 1401


## Section 4 — Distribution of Spatial OOD Scores

In [4]:
import plotly.graph_objects as go

# Load whole-image embeddings from DB
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    'SELECT img_name, split, sequence, spatial_mean_embedding FROM embeddings ORDER BY id'
).fetchall()
conn.close()
names_all  = [r[0] for r in rows]
splits_all = [r[1] for r in rows]
seqs_all   = [r[2] for r in rows]
embeds_all = np.stack([np.frombuffer(r[3], dtype=np.float32) for r in rows])

normed     = embeds_all / (np.linalg.norm(embeds_all, axis=1, keepdims=True) + 1e-8)
train_mask = np.array([s=='train' for s in splits_all])
tv         = normed[train_mask]
wms        = np.full(len(splits_all), np.nan)
for i in range(0, len(splits_all), 256):
    wms[i:i+256] = (normed[i:i+256] @ tv.T).max(axis=1)
wms[train_mask] = np.nan

df_all = pd.DataFrame({'img_name': names_all, 'split': splits_all,
                        'sequence': seqs_all,  'whole_max_sim': wms})

# Heatmap stats
hm_stats = []
for rec in eval_records:
    hp = HEATMAPS_DIR / f'{rec["name"]}.npy'
    if hp.exists():
        hm = np.load(hp)
        hm_stats.append({'img_name': rec['name'], 'split': rec['split'],
                          'sequence': rec['sequence'],
                          'mean_patch_sim': float(hm.mean()),
                          'min_patch_sim':  float(hm.min()),
                          'pct_ood_08':     float((hm<0.8).mean()*100),
                          'heatmap_shape':  hm.shape})

df_hm     = pd.DataFrame(hm_stats)
df_merged = df_hm.merge(df_all[['img_name','whole_max_sim']], on='img_name', how='left')
log.info('Heatmap stats: %d images', len(df_hm))

for sp in ['test','val']:
    sub = df_merged[df_merged['split']==sp]
    log.info('%s: N=%d  patch_sim_median=%.4f  whole_sim_median=%.4f  pct<0.8_median=%.1f%%',
             sp.upper(), len(sub), sub['mean_patch_sim'].median(),
             sub['whole_max_sim'].median(), sub['pct_ood_08'].median())

print(df_merged.groupby('split')[['mean_patch_sim','whole_max_sim','pct_ood_08']].describe().round(4))

2026-03-02 23:39:28,040 INFO Heatmap stats: 1401 images


2026-03-02 23:39:28,046 INFO TEST: N=1203  patch_sim_median=0.8079  whole_sim_median=0.9108  pct<0.8_median=40.2%


2026-03-02 23:39:28,049 INFO VAL: N=198  patch_sim_median=0.8083  whole_sim_median=0.9206  pct<0.8_median=40.7%


      mean_patch_sim                                                          \
               count    mean     std     min     25%     50%     75%     max   
split                                                                          
test          1203.0  0.8007  0.0635  0.6039  0.7524  0.8079  0.8469  0.9274   
val            198.0  0.8054  0.0580  0.6950  0.7609  0.8083  0.8463  0.9367   

      whole_max_sim         ...                 pct_ood_08                    \
              count   mean  ...     75%     max      count     mean      std   
split                       ...                                                
test         1203.0  0.898  ...  0.9491  0.9979     1203.0  45.4610  32.4477   
val           198.0  0.904  ...  0.9486  0.9924      198.0  44.1155  31.1936   

                                                    
          min      25%      50%      75%       max  
split                                               
test   0.0245  15.2569  40.2500  76.302

In [5]:
fig4a = go.Figure()
for sp, col in [('test','#2ca02c'),('val','#ff7f0e')]:
    sub = df_merged[df_merged['split']==sp]
    fig4a.add_trace(go.Histogram(x=sub['mean_patch_sim'],name=sp,opacity=0.65,
                                  nbinsx=50,marker_color=col))
fig4a.update_layout(barmode='overlay',
    title='Mean patch cosine similarity to training (per image)',
    xaxis_title='Mean max cosine sim (patches)', yaxis_title='Images',
    width=850, height=450)
fig4a.show()

fig4b = go.Figure()
for sp, col in [('test','#2ca02c'),('val','#ff7f0e')]:
    sub = df_merged[df_merged['split']==sp]
    fig4b.add_trace(go.Scatter(x=sub['whole_max_sim'], y=sub['mean_patch_sim'],
        mode='markers', name=sp, marker=dict(size=4,color=col,opacity=0.6),
        text=sub['img_name'],
        hovertemplate='<b>%{text}</b><br>whole=%{x:.4f}<br>patch=%{y:.4f}<extra></extra>'))
fig4b.update_layout(title='Whole-image vs mean-patch cosine similarity',
    xaxis_title='Whole-image max cosine sim',
    yaxis_title='Mean patch max cosine sim', width=700, height=500)
fig4b.show()

from scipy.stats import spearmanr
tc = df_merged[df_merged['split']=='test'].dropna(subset=['whole_max_sim','mean_patch_sim'])
rho, pval = spearmanr(tc['whole_max_sim'], tc['mean_patch_sim'])
log.info('Spearman rho (whole vs patch, test): %.4f  p=%.2e', rho, pval)
print(f'Spearman rho: {rho:.4f}  p={pval:.2e}')

2026-03-02 23:39:30,612 INFO Spearman rho (whole vs patch, test): 0.8555  p=0.00e+00


Spearman rho: 0.8555  p=0.00e+00


## Section 5 — Top-20 Most OOD Test Images: Spatial Heatmaps

In [6]:
df_test        = df_merged[df_merged['split']=='test'].copy()
df_test_sorted = df_test.sort_values('mean_patch_sim')
top20_ood      = df_test_sorted.head(20)

rng         = random.Random(42)
top10pct    = df_test_sorted.tail(max(1, len(df_test_sorted)//10))
sample_good = df_merged.loc[rng.sample(list(top10pct.index), min(5,len(top10pct)))]

log.info('Top-20 most OOD test images:')
for _, row in top20_ood.iterrows():
    log.info('  %s  patch_sim=%.4f  whole_sim=%.4f',
             row['img_name'], row['mean_patch_sim'],
             row.get('whole_max_sim', float('nan')))

def make_overlay_figure(rec_row, save_path):
    img_path = DATASET_ROOT / rec_row['split'] / 'images' / f"{rec_row['img_name']}.jpg"
    hm_path  = HEATMAPS_DIR / f"{rec_row['img_name']}.npy"
    if not img_path.exists() or not hm_path.exists(): return
    img    = Image.open(img_path).convert('RGB')
    hm     = np.load(hm_path)
    W, H   = img.size
    hm_up  = np.array(Image.fromarray(hm).resize((W,H), Image.BILINEAR))
    fig, ax = plt.subplots(1, 2, figsize=(14,5))
    ax[0].imshow(np.array(img)); ax[0].set_title('Original'); ax[0].axis('off')
    ax[1].imshow(np.array(img))
    vmin = max(0.0, float(hm.min())-0.05); vmax = min(1.0, float(hm.max())+0.05)
    im = ax[1].imshow(hm_up, cmap='RdYlGn', alpha=0.55, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax[1], shrink=0.8, label='Max cosine sim to training')
    ax[1].set_title(f'OOD heatmap  mean={rec_row["mean_patch_sim"]:.4f}'); ax[1].axis('off')
    fig.suptitle(f"{rec_row['img_name']}  ({rec_row['split']})", fontsize=11, y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=100, bbox_inches='tight'); plt.close(fig)

for _, row in tqdm(top20_ood.iterrows(), total=20, desc='OOD overlays'):
    make_overlay_figure(row, FIGURES_DIR / f"{row['img_name']}.png")
for _, row in tqdm(sample_good.iterrows(), total=len(sample_good), desc='Good overlays'):
    make_overlay_figure(row, FIGURES_DIR / f"{row['img_name']}_good.png")
log.info('Individual overlay figures saved.')

2026-03-02 23:39:30,634 INFO Top-20 most OOD test images:


2026-03-02 23:39:30,635 INFO   orca_usv_inland_N03_2_01837  patch_sim=0.6039  whole_sim=0.7821


2026-03-02 23:39:30,636 INFO   yt017_00_00269  patch_sim=0.6342  whole_sim=0.7325


2026-03-02 23:39:30,636 INFO   inhouse_seq_111_00642  patch_sim=0.6477  whole_sim=0.7099


2026-03-02 23:39:30,637 INFO   yt017_00_00154  patch_sim=0.6522  whole_sim=0.7202


...

2026-03-02 23:39:30,648 INFO   inhouse_seq_111_01550  patch_sim=0.6770  whole_sim=0.7060


2026-03-02 23:39:30,648 INFO   yt007_03_00800  patch_sim=0.6779  whole_sim=0.7568


OOD overlays:   0%|          | 0/20 [00:00<?, ?it/s]

Good overlays:   0%|          | 0/5 [00:00<?, ?it/s]

2026-03-02 23:39:46,702 INFO Individual overlay figures saved.


In [7]:
fig_grid, axes = plt.subplots(4, 5, figsize=(20, 16))
for ai, (_, row) in enumerate(top20_ood.iterrows()):
    ax  = axes.flatten()[ai]
    ipath = DATASET_ROOT / row['split'] / 'images' / f"{row['img_name']}.jpg"
    hpath = HEATMAPS_DIR / f"{row['img_name']}.npy"
    if ipath.exists() and hpath.exists():
        img = Image.open(ipath).convert('RGB'); hm = np.load(hpath)
        W,H = img.size
        ax.imshow(np.array(img))
        ax.imshow(np.array(Image.fromarray(hm).resize((W,H),Image.BILINEAR)),
                  cmap='RdYlGn', alpha=0.55,
                  vmin=max(0.,float(hm.min())-0.05), vmax=min(1.,float(hm.max())+0.05))
        nm = row['img_name'][-25:] if len(row['img_name'])>25 else row['img_name']
        ax.set_title(f'{nm}\n{row["mean_patch_sim"]:.3f}', fontsize=7)
    ax.axis('off')
fig_grid.suptitle('Top-20 Most Spatially-OOD Test Images (green=covered, red=OOD)',
                  fontsize=13, y=1.01)
plt.tight_layout()
grid_path = FIGURES_DIR / 'top20_ood_grid.png'
fig_grid.savefig(grid_path, dpi=100, bbox_inches='tight')
plt.close(fig_grid)
log.info('Summary grid: %s', grid_path)
print(f'Grid: {grid_path}')

2026-03-02 23:39:51,917 INFO Summary grid: /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/figures/top20_ood_grid.png


Grid: /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/figures/top20_ood_grid.png


## Section 6 — Aggregate Spatial OOD Patterns

In [8]:
shapes = [r['heatmap_shape'] for _,r in df_merged[df_merged['split']=='test'].iterrows()]
GRID_H, GRID_W = Counter(shapes).most_common(1)[0][0]
log.info('Common grid: %dx%d', GRID_H, GRID_W)

agg, cnt = np.zeros((GRID_H,GRID_W), np.float64), 0
for _,row in tqdm(df_merged[df_merged['split']=='test'].iterrows(),
                  total=(df_merged['split']=='test').sum(), desc='Aggregating'):
    hp = HEATMAPS_DIR / f'{row["img_name"]}.npy'
    if hp.exists():
        hm = np.load(hp)
        if hm.shape != (GRID_H,GRID_W):
            hm = np.array(Image.fromarray(hm).resize((GRID_W,GRID_H),Image.BILINEAR))
        agg += hm; cnt += 1
agg_mean = (agg/max(cnt,1)).astype(np.float32)
np.save(SPATIAL_DIR/'aggregate_heatmap.npy', agg_mean)
log.info('Aggregated %d images', cnt)

fig6, ax6 = plt.subplots(figsize=(12,6))
im6 = ax6.imshow(agg_mean, cmap='RdYlGn', aspect='auto')
plt.colorbar(im6, ax=ax6, label='Mean max cosine sim to training')
ax6.set_title(f'Aggregate spatial OOD heatmap — all test images (n={cnt})', fontsize=13)
ax6.set_xlabel(f'Patch col  (grid: {GRID_W} wide, 1 cell=16px)')
ax6.set_ylabel(f'Patch row  (grid: {GRID_H} tall, 1 cell=16px)')
plt.tight_layout()
agg_fig = FIGURES_DIR / 'aggregate_heatmap.png'
fig6.savefig(agg_fig, dpi=120, bbox_inches='tight')
plt.close(fig6)
log.info('Aggregate figure: %s', agg_fig)

rm = agg_mean.mean(axis=1); cm = agg_mean.mean(axis=0)
log.info('Most OOD row %d/%d (%.4f)  col %d/%d (%.4f)',
         int(rm.argmin()), GRID_H, float(rm.min()),
         int(cm.argmin()), GRID_W, float(cm.min()))
print(f'Most OOD row: {int(rm.argmin())}/{GRID_H}  col: {int(cm.argmin())}/{GRID_W}')

2026-03-02 23:39:51,973 INFO Common grid: 68x120


Aggregating:   0%|          | 0/1203 [00:00<?, ?it/s]

2026-03-02 23:39:52,287 INFO Aggregated 1203 images


2026-03-02 23:39:52,511 INFO Aggregate figure: /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/figures/aggregate_heatmap.png


2026-03-02 23:39:52,514 INFO Most OOD row 16/68 (0.7871)  col 119/120 (0.7627)


Most OOD row: 16/68  col: 119/120


In [9]:
seq_sim = df_merged[df_merged['split']=='test'].groupby('sequence')['mean_patch_sim'].mean()
top5    = seq_sim.nsmallest(5).index.tolist()
log.info('Top-5 OOD sequences: %s', top5)

fig6s, axs = plt.subplots(1, 5, figsize=(20,4))
for ax_s, sq in zip(axs, top5):
    imgs = df_merged[(df_merged['split']=='test')&(df_merged['sequence']==sq)]
    ss, sc = np.zeros((GRID_H,GRID_W), np.float64), 0
    for _,r in imgs.iterrows():
        hp = HEATMAPS_DIR / f'{r["img_name"]}.npy'
        if hp.exists():
            hm = np.load(hp)
            if hm.shape!=(GRID_H,GRID_W):
                hm = np.array(Image.fromarray(hm).resize((GRID_W,GRID_H),Image.BILINEAR))
            ss += hm; sc += 1
    if sc:
        im_s = ax_s.imshow((ss/sc).astype(np.float32), cmap='RdYlGn', aspect='auto',
                            vmin=float(agg_mean.min()), vmax=float(agg_mean.max()))
        plt.colorbar(im_s, ax=ax_s, shrink=0.8)
    ax_s.set_title(f'{sq}\n(n={sc}, {seq_sim[sq]:.3f})', fontsize=8); ax_s.axis('off')
fig6s.suptitle('Per-sequence aggregate heatmaps — top-5 most OOD test sequences', fontsize=12)
plt.tight_layout()
seq_fig = FIGURES_DIR / 'top5_seq_heatmaps.png'
fig6s.savefig(seq_fig, dpi=100, bbox_inches='tight'); plt.close(fig6s)
log.info('Seq heatmaps: %s', seq_fig)

2026-03-02 23:39:52,534 INFO Top-5 OOD sequences: ['yt026_07', 'yt017_00', 'yt049_02', 'yt007_03', 'yt050_01']


2026-03-02 23:39:53,008 INFO Seq heatmaps: /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/figures/top5_seq_heatmaps.png


## Section 7 — Report Generation

In [10]:
import subprocess

log.info('Generating spatial OOD PDF report...')
res = subprocess.run(
    [sys.executable, str(REPORT_GEN),
     '--heatmaps', str(HEATMAPS_DIR),
     '--figures',  str(FIGURES_DIR),
     '--db',       str(DB_PATH),
     '--images',   str(DATASET_ROOT),
     '--out',      str(REPORT_PATH)],
    capture_output=True, text=True)

if res.returncode == 0:
    log.info('Report: %s', REPORT_PATH)
    print(res.stdout[-600:])
else:
    log.error('Report failed: %s', res.stderr[-1000:])
    print('STDERR:', res.stderr[-600:])

log.info('=== spatial_ood.ipynb execution complete ===')

2026-03-02 23:39:53,023 INFO Generating spatial OOD PDF report...


2026-03-02 23:39:59,744 INFO Report: /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/spatial_report.pdf


tistics from /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/heatmaps ...
  Heatmaps loaded: 1401 images
Loading embeddings from /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/embeddings.db ...
  Merged dataframe: 1401 rows
Building HTML report ...
Generating figures for report...
  HTML written to /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/spatial_report.html
Converting to PDF via weasyprint ...
  PDF written to /home/farms/macvi2026-farms-team/challenges/common/lars-analysis/spatial_ood/spatial_report.pdf
Done.

2026-03-02 23:39:59,746 INFO === spatial_ood.ipynb execution complete ===
